# 09 — Spatial Validation

## Objective

This notebook evaluates the spatial characteristics of the 30 m LST downscaled products using additional spatial validation measures.

The analysis is designed to determine whether Random Forest and XGBoost preserve meaningful spatial structure in the downscaled LST products rather than simply producing smoother temperature distributions.

### Analyses

1. Moran's I — spatial autocorrelation
2. Variogram analysis — spatial dependence across distance
3. Hotspot preservation — spatial agreement of thermal extremes
4. Thermal variability and smoothing assessment

## 1. Moran's I — Spatial Autocorrelation

In [39]:
!pip install libpysal esda


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [40]:
import numpy as np
from pathlib import Path
import rasterio
from libpysal.weights import lat2W
from esda.moran import Moran

In [43]:
PROJECT_DIR = Path(r"D:\Projects\GeoAI-LST-Downscaling")
FINE_DIR = PROJECT_DIR / "data" / "processed" / "lucknow" / "fine"
COARSE_DIR = PROJECT_DIR / "data" / "processed" / "lucknow" / "coarse"
DOWNSCALED_DIR = PROJECT_DIR / "data" / "processed" / "lucknow" / "downscaled"

reference_path = FINE_DIR/ "predictors" / "lst_clean_lucknow.tif"
tsharp_path = DOWNSCALED_DIR / "tsharp_lst_30m_lucknow.tif"
rf_path = DOWNSCALED_DIR / "rf_lst_30m_lucknow.tif"
xgb_path = DOWNSCALED_DIR / "xgb_lst_30m_lucknow.tif"

In [44]:




def read_raster(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Raster not found: {path}")

    with rasterio.open(path) as src:
        data = src.read(1).astype(np.float64)
        profile = src.profile.copy()
        nodata = src.nodata

    # Convert NoData values to NaN
    if nodata is not None and not np.isnan(nodata):
        data[data == nodata] = np.nan

    return data, profile, nodata

reference, ref_profile, ref_nodata = read_raster(reference_path)
tsharp, tsharp_profile, tsharp_nodata = read_raster(tsharp_path)
rf, rf_profile, rf_nodata = read_raster(rf_path)
xgb, xgb_profile, xgb_nodata = read_raster(xgb_path)

print("Reference:", reference.shape)
print("TsHARP:", tsharp.shape)
print("RF:", rf.shape)
print("XGBoost:", xgb.shape)

Reference: (703, 762)
TsHARP: (702, 762)
RF: (703, 762)
XGBoost: (703, 762)


In [48]:
print("Reference shape:", reference.shape)
print("TsHARP shape:", tsharp.shape)
print("RF shape:", rf.shape)
print("XGBoost shape:", xgb.shape)

print("Reference resolution:", ref_profile["transform"].a, abs(ref_profile["transform"].e))
print("TsHARP resolution:", tsharp_profile["transform"].a, abs(tsharp_profile["transform"].e))
print("RF resolution:", rf_profile["transform"].a, abs(rf_profile["transform"].e))
print("XGBoost resolution:", xgb_profile["transform"].a, abs(xgb_profile["transform"].e))



Reference shape: (703, 762)
TsHARP shape: (702, 762)
RF shape: (703, 762)
XGBoost shape: (703, 762)
Reference resolution: 30.0 30.0
TsHARP resolution: 30.0 30.0
RF resolution: 30.0 30.0
XGBoost resolution: 30.0 30.0


In [54]:
for name, (data, profile) in rasters.items():
    print(f"\n{name}")
    print("Shape:", data.shape)
    print("Transform:", profile["transform"])
    print("Bounds:", rasterio.transform.array_bounds(
        data.shape[0],
        data.shape[1],
        profile["transform"]
    ))


Reference
Shape: (703, 762)
Transform: | 30.00, 0.00, 483225.00|
| 0.00,-30.00, 2979165.00|
| 0.00, 0.00, 1.00|
Bounds: (483225.0, 2958075.0, 506085.0, 2979165.0)

TsHARP
Shape: (702, 762)
Transform: | 30.00, 0.00, 483225.00|
| 0.00,-30.00, 2979165.00|
| 0.00, 0.00, 1.00|
Bounds: (483225.0, 2958105.0, 506085.0, 2979165.0)

RF
Shape: (703, 762)
Transform: | 30.00, 0.00, 483225.00|
| 0.00,-30.00, 2979165.00|
| 0.00, 0.00, 1.00|
Bounds: (483225.0, 2958075.0, 506085.0, 2979165.0)

XGBoost
Shape: (703, 762)
Transform: | 30.00, 0.00, 483225.00|
| 0.00,-30.00, 2979165.00|
| 0.00, 0.00, 1.00|
Bounds: (483225.0, 2958075.0, 506085.0, 2979165.0)


In [55]:
print("\nCRS:")
for name, (_, profile) in rasters.items():
    print(name, profile["crs"])

print("\nResolution:")
for name, (_, profile) in rasters.items():
    print(
        name,
        profile["transform"].a,
        abs(profile["transform"].e)
    )


CRS:
Reference EPSG:32644
TsHARP EPSG:32644
RF EPSG:32644
XGBoost EPSG:32644

Resolution:
Reference 30.0 30.0
TsHARP 30.0 30.0
RF 30.0 30.0
XGBoost 30.0 30.0


In [56]:
# Crop all rasters to the common spatial extent
common_rows = min(
    reference.shape[0],
    tsharp.shape[0],
    rf.shape[0],
    xgb.shape[0]
)

common_cols = min(
    reference.shape[1],
    tsharp.shape[1],
    rf.shape[1],
    xgb.shape[1]
)

reference_common = reference[:common_rows, :common_cols]
tsharp_common = tsharp[:common_rows, :common_cols]
rf_common = rf[:common_rows, :common_cols]
xgb_common = xgb[:common_rows, :common_cols]

print("Reference:", reference_common.shape)
print("TsHARP:", tsharp_common.shape)
print("RF:", rf_common.shape)
print("XGBoost:", xgb_common.shape)

Reference: (702, 762)
TsHARP: (702, 762)
RF: (702, 762)
XGBoost: (702, 762)


In [57]:
valid_mask = (
    np.isfinite(reference_common) &
    np.isfinite(tsharp_common) &
    np.isfinite(rf_common) &
    np.isfinite(xgb_common)
)

print("Total pixels:", valid_mask.size)
print("Valid pixels:", valid_mask.sum())
print("Invalid pixels:", (~valid_mask).sum())

Total pixels: 534924
Valid pixels: 314648
Invalid pixels: 220276


In [61]:
reference_values = reference_common[valid_mask]
tsharp_values = tsharp_common[valid_mask]
rf_values = rf_common[valid_mask]
xgb_values = xgb_common[valid_mask]

print("Number of valid pixels:", len(reference_values))

Number of valid pixels: 314648


In [62]:
from libpysal.weights import lat2W

rows, cols = reference.shape

# Rook contiguity: edge-sharing neighbours
w = lat2W(rows, cols, rook=True)
# Create rook-contiguity weights for the complete raster grid
w_full = lat2W(rows, cols, rook=True)
print("Number of spatial units:", w.n)
print("Number of neighbours for first pixel:", len(w.neighbors[0]))

Number of spatial units: 535686
Number of neighbours for first pixel: 2


In [63]:
print("Total pixels:", reference.size)
print("Valid pixels:", valid_mask.sum())
print("Invalid pixels:", (~valid_mask).sum())

Total pixels: 535686
Valid pixels: 314648
Invalid pixels: 220276


In [64]:
valid_flat = valid_mask.flatten()

valid_indices = np.where(valid_flat)[0]

print("Valid spatial units:", len(valid_indices))

Valid spatial units: 314648


In [65]:
from libpysal.weights import W

valid_indices = np.where(valid_flat)[0]

# Map original raster index -> new index
index_map = {
    old_idx: new_idx
    for new_idx, old_idx in enumerate(valid_indices)
}

neighbors = {}
weights = {}

for old_idx in valid_indices:
    new_idx = index_map[old_idx]

    # Original neighbours from the full raster
    old_neighbors = w_full.neighbors[old_idx]

    # Keep only neighbours that are also valid
    valid_neighbors = [
        n for n in old_neighbors
        if n in index_map
    ]

    neighbors[new_idx] = [
        index_map[n] for n in valid_neighbors
    ]

    weights[new_idx] = [1.0] * len(valid_neighbors)

# Create the reduced spatial weights object
w = W(neighbors, weights)

print("Original spatial units:", w_full.n)
print("Valid spatial units:", len(valid_indices))
print("Spatial units in new weights:", w.n)

Original spatial units: 535686
Valid spatial units: 314648
Spatial units in new weights: 314648


In [66]:
print("Number of islands:", w.n_components)
print("Average neighbours:", np.mean(list(map(len, w.neighbors.values()))))

Number of islands: 1
Average neighbours: 3.9876941852482775


In [68]:
reference_values = reference_common[valid_mask]
tsharp_values = tsharp_common[valid_mask]
rf_values = rf_common[valid_mask]
xgb_values = xgb_common[valid_mask]

In [69]:
from esda.moran import Moran

# Calculate Moran's I
moran_reference = Moran(
    reference_values,
    w,
    permutations=999
)

moran_tsharp = Moran(
    tsharp_values,
    w,
    permutations=999
)

moran_rf = Moran(
    rf_values,
    w,
    permutations=999
)

moran_xgb = Moran(
    xgb_values,
    w,
    permutations=999
)

In [70]:
print("Reference 30 m")
print("Moran's I:", moran_reference.I)
print("Expected I:", moran_reference.EI)
print("Z-score:", moran_reference.z_sim)
print("Permutation p-value:", moran_reference.p_sim)

print("\nTsHARP 30 m")
print("Moran's I:", moran_tsharp.I)
print("Expected I:", moran_tsharp.EI)
print("Z-score:", moran_tsharp.z_sim)
print("Permutation p-value:", moran_tsharp.p_sim)

print("\nRandom Forest 30 m")
print("Moran's I:", moran_rf.I)
print("Expected I:", moran_rf.EI)
print("Z-score:", moran_rf.z_sim)
print("Permutation p-value:", moran_rf.p_sim)

print("\nXGBoost 30 m")
print("Moran's I:", moran_xgb.I)
print("Expected I:", moran_xgb.EI)
print("Z-score:", moran_xgb.z_sim)
print("Permutation p-value:", moran_xgb.p_sim)

Reference 30 m
Moran's I: 0.9842988877260397
Expected I: -3.178164736990977e-06
Z-score: 773.9710173474639
Permutation p-value: 0.001

TsHARP 30 m
Moran's I: 0.9438660487743782
Expected I: -3.178164736990977e-06
Z-score: 756.5420596448761
Permutation p-value: 0.001

Random Forest 30 m
Moran's I: 0.8138341868665738
Expected I: -3.178164736990977e-06
Z-score: 655.151754275072
Permutation p-value: 0.001

XGBoost 30 m
Moran's I: 0.828310805810523
Expected I: -3.178164736990977e-06
Z-score: 646.7128826139821
Permutation p-value: 0.001


In [76]:
import pandas as pd

RESULTS_DIR = PROJECT_DIR / "results" / "spatial_validation"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

moran_results = {
    "Reference": {
        "Moran_I": moran_reference.I,
        "Expected_I": moran_reference.EI,
        "Z_score": moran_reference.z_sim,
        "p_value": moran_reference.p_sim,
    },
    "TsHARP": {
        "Moran_I": moran_tsharp.I,
        "Expected_I": moran_tsharp.EI,
        "Z_score": moran_tsharp.z_sim,
        "p_value": moran_tsharp.p_sim,
    },
    "Random Forest": {
        "Moran_I": moran_rf.I,
        "Expected_I": moran_rf.EI,
        "Z_score": moran_rf.z_sim,
        "p_value": moran_rf.p_sim,
    },
    "XGBoost": {
        "Moran_I": moran_xgb.I,
        "Expected_I": moran_xgb.EI,
        "Z_score": moran_xgb.z_sim,
        "p_value": moran_xgb.p_sim,
    },
}

moran_df = pd.DataFrame.from_dict(moran_results, orient="index")

moran_output = RESULTS_DIR / "moran_i_results.csv"
moran_df.to_csv(moran_output, index_label="Method")

print(f"Moran's I results saved to:\n{moran_output}")

moran_df

Moran's I results saved to:
D:\Projects\GeoAI-LST-Downscaling\results\spatial_validation\moran_i_results.csv


,Moran_I,Expected_I,Z_score,p_value
Reference,0.984299,-0.000003,773.971017,0.001
TsHARP,0.943866,-0.000003,756.542060,0.001
Random Forest,0.813834,-0.000003,655.151754,0.001
XGBoost,0.828311,-0.000003,646.712883,0.001
